## 1. Importing Libraries

In [ ]:
# Import the libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
pd.set_option("display.max_columns", None)

## 2. Reading the Dataset

In [ ]:
# Read the original dataset
original_DF = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
# Display the first few records
original_DF.head()

In [ ]:
# Display the number of rows and columns
original_DF.shape

In [ ]:
# Display the data types of each column
original_DF.dtypes

In [ ]:
# Display summary statistics
original_DF.describe(include="all")

## 3. Data Cleaning and Preprocessing

### 3.1 Checking Duplicate Records

In [ ]:
# Check duplicate rows
print("Number of duplicate rows:", original_DF.duplicated().sum())

# Check duplicate customer IDs
print("Number of duplicate customer IDs:", original_DF["customerID"].duplicated().sum())

### 3.2 Checking Missing Values

In [ ]:
# Check missing values in the original dataset
original_DF.isnull().sum()

In [ ]:
# Check whitespace-only values
for column in original_DF.select_dtypes(include="object").columns:
    whitespace_count = original_DF[column].astype(str).str.strip().eq("").sum()
    if whitespace_count > 0:
        print(f"{column}: {whitespace_count} whitespace-only values")

### 3.3 Creating the Cleansed Dataset

In [ ]:
# Create a copy of the original dataset
cleansed_DF = original_DF.copy()

# Remove leading and trailing whitespace from text columns
text_cols = cleansed_DF.select_dtypes(include="object").columns

for column in text_cols:
    cleansed_DF[column] = cleansed_DF[column].str.strip()

# Convert TotalCharges into a numeric column
cleansed_DF["TotalCharges"] = pd.to_numeric(
    cleansed_DF["TotalCharges"].replace("", np.nan),
    errors="coerce"
)

cleansed_DF.head()

### 3.4 Handling Missing Total Charges

In [ ]:
# Identify records where TotalCharges is missing
missing_total_charges = cleansed_DF["TotalCharges"].isnull()

print("Missing TotalCharges values:", missing_total_charges.sum())
print("Missing TotalCharges with zero tenure:",
      (missing_total_charges & (cleansed_DF["tenure"] == 0)).sum())
print("Missing TotalCharges with tenure above zero:",
      (missing_total_charges & (cleansed_DF["tenure"] > 0)).sum())

# Preserve an indicator of the originally missing values
cleansed_DF["total_charges_was_missing"] = missing_total_charges.astype(int)

# The 11 missing values belong to customers with zero tenure.
# Set TotalCharges to 0 for these records because no completed service tenure is recorded.
cleansed_DF.loc[
    missing_total_charges & (cleansed_DF["tenure"] == 0),
    "TotalCharges"
] = 0

print("Remaining missing values:", cleansed_DF["TotalCharges"].isnull().sum())

### 3.5 Checking Inconsistent Entries

In [ ]:
# Check unique values in categorical columns
categorical_cols = cleansed_DF.select_dtypes(include="object").columns

for column in categorical_cols:
    print(f"\\n{column}")
    print(cleansed_DF[column].value_counts(dropna=False))

In [ ]:
# Check expected values in selected categorical columns
expected_values = {
    "gender": {"Female", "Male"},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "PaperlessBilling": {"Yes", "No"},
    "Churn": {"Yes", "No"}
}

for column, valid_values in expected_values.items():
    invalid_values = cleansed_DF.loc[
        ~cleansed_DF[column].isin(valid_values), column
    ].unique()
    print(f"{column} - unexpected values: {invalid_values}")

In [ ]:
# Check numeric values and ranges
print("Negative tenure values:", (cleansed_DF["tenure"] < 0).sum())
print("Negative MonthlyCharges values:", (cleansed_DF["MonthlyCharges"] < 0).sum())
print("Negative TotalCharges values:", (cleansed_DF["TotalCharges"] < 0).sum())
print("Unexpected SeniorCitizen values:",
      sorted(set(cleansed_DF["SeniorCitizen"].dropna().unique()) - {0, 1}))

## 4. Feature Engineering

In [ ]:
# Create a tenure group
cleansed_DF["tenure_group"] = pd.cut(
    cleansed_DF["tenure"],
    bins=[-1, 12, 24, 48, 60, np.inf],
    labels=["0-12 months", "13-24 months", "25-48 months",
            "49-60 months", "61+ months"]
)

# Count the number of subscribed services
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]

cleansed_DF["total_services"] = cleansed_DF[service_cols].eq("Yes").sum(axis=1)

# Create a binary churn variable
cleansed_DF["churn_flag"] = cleansed_DF["Churn"].map({"No": 0, "Yes": 1})

# Create a binary feature for customers with at least 24 months of tenure
cleansed_DF["long_term_customer"] = (cleansed_DF["tenure"] >= 24).astype(int)

# Display the updated dataset
cleansed_DF.head()

### 4.1 Checking the Cleansed Dataset

In [ ]:
# Check missing values after cleaning
cleansed_DF.isnull().sum()

In [ ]:
# Check duplicate rows after cleaning
print("Duplicate rows:", cleansed_DF.duplicated().sum())
print("Duplicate customer IDs:", cleansed_DF["customerID"].duplicated().sum())

In [ ]:
# Check categorical value frequencies
for column in ["gender", "Contract", "InternetService", "PaymentMethod", "Churn"]:
    print(f"\\n{column}")
    print(cleansed_DF[column].value_counts())

In [ ]:
# Check numerical ranges
numerical_cols = [
    "tenure", "MonthlyCharges", "TotalCharges", "total_services"
]

for column in numerical_cols:
    print(f"\\n{column}")
    print(cleansed_DF[column].describe())

## 4. Exploratory Data Analysis (EDA)

Exploratory Data Analysis is used to understand the distribution of variables and identify patterns, relationships and differences within the dataset.

### 4.1 Univariate Analysis

Univariate analysis examines one variable at a time.

#### 4.1.1 Univariate Analysis of Categorical Variables

In [ ]:
# Univariate analysis for categorical variables
categorical_analysis_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "InternetService", "Contract", "PaymentMethod", "Churn"
]

for column in categorical_analysis_cols:
    plt.figure(figsize=(12, 8))
    cleansed_DF[column].value_counts().plot(kind="bar")
    plt.xlabel(column)
    plt.ylabel("Number of Customers")
    plt.title(f"Distribution of {column}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

#### 4.1.2 Univariate Analysis of Numerical Variables

In [ ]:
# Histograms of numerical variables
cleansed_DF[numerical_cols].hist(figsize=(12, 8), bins=20)
plt.suptitle("Distribution of Numerical Variables")
plt.tight_layout()
plt.show()

In [ ]:
# Churn distribution
plt.figure(figsize=(12, 8))
cleansed_DF["Churn"].value_counts().plot(kind="bar")
plt.xlabel("Customer Churn")
plt.ylabel("Number of Customers")
plt.title("Distribution of Customer Churn")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.2 Bivariate Analysis

Bivariate analysis examines two variables at a time to identify relationships, patterns and differences.

#### 4.2.1 Numerical vs Numerical

In [ ]:
# Tenure vs Monthly Charges
plt.figure(figsize=(12, 8))
plt.scatter(cleansed_DF["tenure"], cleansed_DF["MonthlyCharges"], alpha=0.5)
plt.xlabel("Tenure (Months)")
plt.ylabel("Monthly Charges")
plt.title("Tenure vs Monthly Charges")
plt.show()

In [ ]:
# Tenure vs Total Charges
plt.figure(figsize=(12, 8))
plt.scatter(cleansed_DF["tenure"], cleansed_DF["TotalCharges"], alpha=0.5)
plt.xlabel("Tenure (Months)")
plt.ylabel("Total Charges")
plt.title("Tenure vs Total Charges")
plt.show()

In [ ]:
# Monthly Charges vs Total Charges
plt.figure(figsize=(12, 8))
plt.scatter(cleansed_DF["MonthlyCharges"], cleansed_DF["TotalCharges"], alpha=0.5)
plt.xlabel("Monthly Charges")
plt.ylabel("Total Charges")
plt.title("Monthly Charges vs Total Charges")
plt.show()

In [ ]:
# Total Services vs Monthly Charges
plt.figure(figsize=(12, 8))
plt.scatter(cleansed_DF["total_services"], cleansed_DF["MonthlyCharges"], alpha=0.5)
plt.xlabel("Number of Services")
plt.ylabel("Monthly Charges")
plt.title("Total Services vs Monthly Charges")
plt.show()

#### 4.2.2 Numerical vs Categorical

In [1]:
# Tenure vs Churn
plt.figure(figsize=(12, 8))
cleansed_DF.boxplot(column="tenure", by="Churn")
plt.xlabel("Customer Churn")
plt.ylabel("Tenure (Months)")
plt.title("Tenure vs Customer Churn")
plt.suptitle("")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Monthly Charges vs Churn
plt.figure(figsize=(12, 8))
cleansed_DF.boxplot(column="MonthlyCharges", by="Churn")
plt.xlabel("Customer Churn")
plt.ylabel("Monthly Charges")
plt.title("Monthly Charges vs Customer Churn")
plt.suptitle("")
plt.show()

In [ ]:
# Total Charges vs Churn
plt.figure(figsize=(12, 8))
cleansed_DF.boxplot(column="TotalCharges", by="Churn")
plt.xlabel("Customer Churn")
plt.ylabel("Total Charges")
plt.title("Total Charges vs Customer Churn")
plt.suptitle("")
plt.show()

In [ ]:
# Monthly Charges vs Contract
plt.figure(figsize=(12, 8))
cleansed_DF.boxplot(column="MonthlyCharges", by="Contract")
plt.xlabel("Contract Type")
plt.ylabel("Monthly Charges")
plt.title("Monthly Charges vs Contract Type")
plt.suptitle("")
plt.show()

### 4.3 Categorical Relationship Analysis

In [ ]:
# Contract vs Churn
contract_churn = pd.crosstab(cleansed_DF["Contract"], cleansed_DF["Churn"])

plt.figure(figsize=(12, 8))
sns.heatmap(contract_churn, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Customer Churn")
plt.ylabel("Contract Type")
plt.title("Contract Type vs Customer Churn")
plt.show()

In [ ]:
# Internet Service vs Churn
internet_churn = pd.crosstab(cleansed_DF["InternetService"], cleansed_DF["Churn"])

plt.figure(figsize=(12, 8))
sns.heatmap(internet_churn, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Customer Churn")
plt.ylabel("Internet Service")
plt.title("Internet Service vs Customer Churn")
plt.show()

In [ ]:
# Payment Method vs Churn
payment_churn = pd.crosstab(cleansed_DF["PaymentMethod"], cleansed_DF["Churn"])

plt.figure(figsize=(12, 8))
sns.heatmap(payment_churn, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Customer Churn")
plt.ylabel("Payment Method")
plt.title("Payment Method vs Customer Churn")
plt.show()

In [ ]:
# Tech Support vs Churn
support_churn = pd.crosstab(cleansed_DF["TechSupport"], cleansed_DF["Churn"])

plt.figure(figsize=(12, 8))
sns.heatmap(support_churn, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Customer Churn")
plt.ylabel("Tech Support")
plt.title("Tech Support vs Customer Churn")
plt.show()

### 4.4 Correlation Analysis

In [ ]:
# Select numerical variables for correlation analysis
correlation_cols = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "total_services", "long_term_customer", "churn_flag"
]

correlation_matrix = cleansed_DF[correlation_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Numerical Variables")
plt.show()

### 4.5 Outlier Analysis

In [ ]:
# Identify potential outliers using the IQR method
outlier_cols = ["tenure", "MonthlyCharges", "TotalCharges", "total_services"]

for column in outlier_cols:
    Q1 = cleansed_DF[column].quantile(0.25)
    Q3 = cleansed_DF[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = cleansed_DF[
        (cleansed_DF[column] < lower_bound) |
        (cleansed_DF[column] > upper_bound)
    ]

    print(f"{column}: {len(outliers)} potential outliers")

In [ ]:
# Boxplots of numerical variables
for column in outlier_cols:
    plt.figure(figsize=(12, 8))
    cleansed_DF.boxplot(column=column)
    plt.ylabel(column)
    plt.title(f"Outlier Analysis of {column}")
    plt.show()

### 4.6 Summary

In [ ]:
# Overall churn rate
overall_churn_rate = cleansed_DF["churn_flag"].mean() * 100
print(f"Overall churn rate: {overall_churn_rate:.2f}%")

# Churn rate by contract type
contract_churn_rate = (
    cleansed_DF.groupby("Contract")["churn_flag"].mean() * 100
).sort_values(ascending=False)

display(contract_churn_rate.to_frame("Churn Rate (%)").round(2))

# Churn rate by tenure group
tenure_churn_rate = (
    cleansed_DF.groupby("tenure_group", observed=False)["churn_flag"].mean() * 100
)

display(tenure_churn_rate.to_frame("Churn Rate (%)").round(2))

## 5. Exporting the Cleansed Dataset

In [ ]:
# Export the final cleansed dataset
cleansed_DF.to_csv("telco_customer_churn_cleaned.csv", index=False)

print("Cleansed dataset exported successfully.")
print("Final dataset shape:", cleansed_DF.shape)